# Étapes GPU du pipeline (Colab) : AutoGluon « extreme », TabICL, LimiX-2

Ce notebook exécute sur Colab les étapes qui demandent un GPU. Il lance les mêmes commandes
qu'en local, avec le même code et le même découpage D/T.

| Section | Contenu | GPU conseillé | Durée |
|---|---|---|---|
| AutoGluon | preset `extreme` (portfolio d'AutoGluon 1.5) : réseaux de neurones (TabM...), modèles de fondation (TabICL, TabPFN, Mitra, TabDPT) et boostings | A100 (≥ 20 Go) | `AG_TIME_H` (4 h par défaut) |
| TabICL | correcteur des boostings, contexte complet | A100 | quelques minutes |
| LimiX-2 | correcteur des boostings | L4 ou A100 | ~2 h 30 sur L4 |

**Utilisation**
1. En local : `python -m src.gpu_bundle`, qui produit `gpu_bundle.zip`.
2. Sur Colab, choisir *Exécution → Modifier le type d'exécution* : GPU **A100**, **Mémoire RAM
   élevée**.
3. Régler les interrupteurs de la cellule 1, puis *Tout exécuter*, et uploader
   `gpu_bundle.zip`.
4. Récupérer `gpu_results.zip` et le décompresser à la racine du dépôt local, puis lancer
   `python -m src.pipeline --steps ensemble`.

L'assemblage final (GES) intègre automatiquement les nouveaux membres (`agx_*` pour AutoGluon
GPU). Il ne leur donne du poids que s'ils améliorent l'ensemble sur les prédictions hors bloc
de D.

AutoGluon ne reprend pas un entraînement interrompu : garder l'onglet ouvert pendant les
`AG_TIME_H` heures. La dernière cellule peut être exécutée à tout moment pour récupérer ce
qui est terminé.

## 1. Réglages

In [ ]:
RUN_AUTOGLUON = True
AG_TIME_H = 4.0          # budget d'AutoGluon (heures)
RUN_TABICL = False       # déjà calculé (tabicl, tabicl_l2, tabicl_l2x)
RUN_LIMIX = False        # déjà calculé (limix_l2)
BASES = "lgbmgr_bag15 catgr_1 ag_LightGBMXT_BAG_L1"
# correcteur enrichi (après la section AutoGluon) : bases + trois modèles AutoGluon GPU
BASES_X = BASES + " agx_CatBoost_BAG_L1 agx_PrepLightGBM_BAG_L1 agx_TabDPT-Turbo_BAG_L1"

## 2. Code et données

In [ ]:
import os, zipfile, json
from pathlib import Path
from google.colab import files

os.makedirs("/content/work", exist_ok=True)
os.chdir("/content/work")
if not Path("gpu_bundle.zip").exists():
    up = files.upload()                                   # choisir gpu_bundle.zip
    Path("gpu_bundle.zip").write_bytes(next(iter(up.values())))
zipfile.ZipFile("gpu_bundle.zip").extractall(".")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. AutoGluon, preset `extreme`

Les réseaux de neurones ne sont autorisés qu'à travers AutoGluon (`--allow-nn`). Chaque modèle
est appris par bagging en 5 blocs : ses prédictions hors bloc sur D entrent dans la
bibliothèque. L'installation prend quelques minutes. AutoGluon tourne dans un processus séparé,
donc aucun redémarrage du noyau n'est nécessaire.

In [ ]:
if RUN_AUTOGLUON:
    !pip -q install -U "autogluon.tabular[all,tabarena]"
    !python -m src.models.autogluon_run --preset extreme --allow-nn --prefix agx_ --time-limit {int(AG_TIME_H * 3600)} --path outputs/regression/autogluon_gpu/model

## 4. TabICL et LimiX-2 (correcteurs des boostings)

In [ ]:
if RUN_TABICL or RUN_LIMIX:
    !pip -q install -U tabicl optuna lightgbm catboost
if RUN_TABICL:
    !python -m src.models.tfm --model tabicl --name tabicl
    !python -m src.models.tfm --model tabicl --name tabicl_l2 --base {BASES}
    !python -m src.models.tfm --model tabicl --name tabicl_l2x --base {BASES_X}
if RUN_LIMIX:
    !git clone -q https://github.com/limix-ldm-ai/LimiX.git /content/LimiX 2>/dev/null || true
    !pip -q install -e /content/LimiX
    !LIMIX_DIR=/content/LimiX python -m src.models.tfm --model limix --name limix_l2 --base {BASES} --budget-h 3

## 5. Résultats (exécutable à tout moment)

In [ ]:
lib = Path("outputs/regression/library")
new = sorted(f.stem for f in lib.glob("*.json")
             if f.stem.startswith("agx_") or f.stem in ("tabicl", "tabicl_l2", "tabicl_l2x", "limix_l2"))
for m in new:
    info = json.loads((lib / f"{m}.json").read_text())
    print(f"{m:40s} RMSE hors bloc sur D = {info['rmse_oof_D']:.2f}")
with zipfile.ZipFile("/content/gpu_results.zip", "w") as z:
    for m in new:
        for ext in ("npz", "json"):
            z.write(lib / f"{m}.{ext}")
    lb = Path("outputs/regression/autogluon_gpu/leaderboard.csv")
    if lb.exists():
        z.write(lb)
files.download("/content/gpu_results.zip")